# Character Extraction

This is Phase 1 of converting OCR'd source to `opend6_tools`.
The objective is to locate the raw source for characters and -- to the extent possible -- extract a complete Character or Creature definition.

This can't be **fully** automated, since the source is riddled with tiny errors in OCR processing.
Sometimes the source needs to be cleaned manually.

It is, therefore, an iterative process that involves a number of steps.
There are traps everywhere, and we need to tip-toe through the data, locating and fixing artifacts of the OCR processing.

Once past the manual cleanup steps, the operation has this as a conceptual outline:

```python
print(character(parse(source_text)))
```

The source text needs to be parsed into details: name, attributes, abilities, equipment.

Then a `opend6_tools.character` DSL definition needs to be created from those details.

Once a reasonably complete DSL is available, the source will be discarded.
There's no sense in fixing spelling errors in the source.

Pragmatically, this is a bit much for a single module or script to do.
Mostly because the parsing is rather complicated, and involves a number of intermediate steps.
Therefore, this notebook is used to create TOML versions of the character definitions.

A second notebook emits the final DSL.

In [1]:
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import deque
import tomli_w

base_dir = Path.cwd().parent

## Step 1: Consistent Character Starts
A character source definition **should** have the following layout:

```
    **name**:
    Agility...
    other detils.
```

Initially, we can only guarantee the presence of "Agility".
The `character_search()` function locates likely character definitions.
Other stuff that surfaces needs to be cleaned up manually to assure consistent formatting.

This output from this function supports manual changes to get the formatting consistent.
Once this manual work is complete, the characters are easier to extract from the source files.

In [2]:
keywords = {"Agility", "agility"}
def character_search(path):
    char_file = p.read_text().splitlines()
    names = []
    starts = [n for n, line in enumerate(char_file) if any(keyword in line for keyword in keywords)]
    fixes = []
    for line in starts:
        start = char_file[line]
        if (pos := start.lower().find('agility')) == 0:
            name = char_file[line-1].strip()
            if name.startswith("**") and name.endswith("**:"):
                pass
            else:
                fixes.append("name format")
        else:
            fixes.append("line break")
            name = char_file[line]
        names.append(name)
    print(f"{path.parent.parent.name:12s} {path.name:22s} {fixes} {names}")

In [3]:
character_paths = base_dir.glob("*/characters/*.src")
for p in character_paths:
    character_search(p)

Settlements  viking_warrior.src     [] ['**Typical Viking Warrior**:']
Settlements  philsopher.src         [] ['**Exceptional Greek Philosopher**:']
Settlements  ladira_almer.src       [] ['**Ladira Almer, Head Priestess**:']
Settlements  mayan_priest.src       [] ['**Typical Priest**:']
Settlements  healfdene.src          [] ['**Healfdene, Thane**:']
Settlements  cachamwri_cabry.src    [] ['**Cachamwri Cabry, Inn Keeper**:']
Settlements  ori_swifthand.src      [] ['**Ori Swifthand, Inn Regular**:']
Settlements  ascara.src             [] ['**Ascra**:']
Settlements  cire.src               [] ['**Cire, Guard**:']
Settlements  dock_worker.src        [] ['**Typical Dock Worker**:']
Settlements  tiu_mali.src           [] ['**Tiu Mali**:']
Settlements  nevest.src             [] ['**Nevest, Guard**:']
Settlements  mukhtar.src            [] ['**Mukhtar, Nomad Leader**:']
Settlements  cernay_avers.src       [] ['**Cernay Avers, Cleric**:']
Settlements  garvin_helot.src       [] ['**Garvin Helot

## Step 2: Extract character as blobs of text

Since each character now has a clear start, we can pull out the character and the following blob of text.

The top-level of parsing is to decompose the text into specific clauses.
These are carefully defined by the rules, and a character, or creature, will generally have the six base attributes, and -- perhaps -- many more clauses of details.

Breaking the clauses apart in the text requires looking for keywords.
We can't use a simple list of these keywords because the base attribute words can also appear inside the definitions for Advantages, Disadvantages, or Special Abilities.
This means the keyword list shifts partway through the character description.

Initially, the base attributes and other attributes are used.
Then there's a state change once any of the other attributes have been found.
After this, only the other attributes and optional characteristics will be treated as keywords.

-   Base Attributes: "Agility", etc.
-   Other Attributes: "Move", "Strength Damage", etc.
-   Optional Characteristics: "Disadvantages", "Advantages", "Special Abilities", "Natural Abilities", and "Equipment"

Within each clause there's a secondary level of parsing.
This is a bit more complicated, and requires breaking the clause into tokens, and then parsing the sequence of tokens into the details for each clause.

Currently, there are a number of token types that can be found.

```peg
word : [<letters> "'"]+
dice : "+"? <digits> "D" ["+" <digits>]?
rank : "(" "R" <digits> <letters-and-digits> ")"
number : "+"? <digits>
punctuation : '"' | "(" | ")" | "/" | ":" | ";" | "," | "." | "-" | "!"
```

The tokenizer yields two-tuples of (token_type, text).
This means parsers can look for `('punctuation', ';')` to spot a specific puncutation mark.
Or, (via the case/match statement) `case ('punctuation', str):` to spot any generic punctuation mark.

The first pass of validation is to decompose each clause into tokens.
This fails when OCR has emitted ``3D+l`` instead of ``3D+1``. The ``3D+l`` is not a recognized token, and the source file needs to be tweaked manually to fix the problem.

There are **numerous** OCR problems that as spotted via an inability to tokenize the source text.

In [4]:
import re

@dataclass
class AttributeClause:
    source: str
    tokens: list[str] = field(init=False, default_factory=list)

    def attr_clause_tokens(self):
        patterns = re.compile(
            r"^(?P<word>[A-Za-z'/-]+)"
            r"|^(?P<dice1>\+?\d+\s*D\s*\+\s*\d+)"
            r"|^(?P<dice2>\d+\s*D)"
            r"|^(?P<rank>\(\s*R\s*\d[\s\w]*\))"
            r"|^(?P<number1>\+\s*\d+)"
            r"|^(?P<number2>\d+)"
            r"|^(?P<punctuation>[\"\(\):;,\.!])"
            r"|\s"
        )
        working = self.source
        while working:
            if match := patterns.match(working):
                if gd_filtered := {name: text for name, text in match.groupdict().items() if text}:
                    yield list(gd_filtered.items())[0]
                working = working[match.end():]
            else:
                pos = len(working)
                raise SyntaxError(f"{self.source[:-pos]!r} ^ {self.source[-pos:]!r}")

    def __post_init__(self):
        self.tokens = list(self.attr_clause_tokens())

base_attributes = {"Agility", "Coordination", "Physique", "Charisma", "Intellect", "Acumen", "Magic", "Miracles"}
other_attributes = {
    "Strength Damage:", "Move:", "Fate Points:", "Character Points:", "Body Points:", "Wound levels:"}
options = {"Disadvantages:?", "Advantages:?", "Special Abilities:?", "Natural Abilities:?", "Equipment:"}

@dataclass
class CharacterText:
    name: str
    details: str
    clauses: list[AttributeClause] = field(init=False, default_factory=list)

    def clause_iter(self):
        """Stateful: base_attributes | other_attributes until first of other_attributes. then other_attributes | options."""
        # Start with base | other
        base_pattern = re.compile("|".join(base_attributes | other_attributes))
        starts = [match.start() for match in base_pattern.finditer(self.details)] + [len(self.details)]
        pairs = zip(starts[:-1], starts[1:])
        base_ending = len(self.details)  # If we never find any "other_attributes" labels...
        for start, end in pairs:
            clause_text = self.details[start:end]
            if any(clause_text.startswith(c) for c in base_attributes):
                yield AttributeClause(clause_text)
            else:
                base_ending = start
                break
        # Continue with other | options
        tail = self.details[base_ending:]
        other_option_pattern = re.compile("|".join(other_attributes | options))
        starts = [match.start() for match in other_option_pattern.finditer(tail)] + [len(tail)]
        pairs = zip(starts[:-1], starts[1:])
        for start, end in pairs:
            yield AttributeClause(tail[start:end])

    def __post_init__(self):
        self.clauses = list(self.clause_iter())

Step 1's cleanup permits the definition of a function to decompose a `.src` file into individual character text blobs.
The `character_iter()` function allows tidy iteration through files with multiple characters.

But only after the source has been tweaked manually to assure the characters are consistently formatted.

In [5]:
def character_text_iter(path):
    char_file = [l.strip() for l in path.read_text().splitlines()]
    starts = [n for n, line in enumerate(char_file) if line.startswith("**") and line.endswith("**:")] + [len(char_file)]
    pairs = zip(starts[:-1], starts[1:])
    for start, end in pairs:
        yield CharacterText(char_file[start][2:-3], ' '.join(char_file[start+1:end]))

An example of an approach to debugging the breakdown of a character into clauses...

In [6]:
fang_src = base_dir / "Other_Places" / "characters" / "fang.src"
fang = list(character_text_iter(fang_src))[0]
print(fang.name)
for c in fang.clauses:
    print(c)

Fang
AttributeClause(source='Agility 4D+1, acrobatics 6D, dodge 7D+1, fighting 7D+2 ', tokens=[('word', 'Agility'), ('dice1', '4D+1'), ('punctuation', ','), ('word', 'acrobatics'), ('dice2', '6D'), ('punctuation', ','), ('word', 'dodge'), ('dice1', '7D+1'), ('punctuation', ','), ('word', 'fighting'), ('dice1', '7D+2')])
AttributeClause(source='Coordination 3D, throwing 3D+1 ', tokens=[('word', 'Coordination'), ('dice2', '3D'), ('punctuation', ','), ('word', 'throwing'), ('dice1', '3D+1')])
AttributeClause(source='Physique 3D, stamina 4D ', tokens=[('word', 'Physique'), ('dice2', '3D'), ('punctuation', ','), ('word', 'stamina'), ('dice2', '4D')])
AttributeClause(source='Intellect 2D: scholar: snakes 4D ', tokens=[('word', 'Intellect'), ('dice2', '2D'), ('punctuation', ':'), ('word', 'scholar'), ('punctuation', ':'), ('word', 'snakes'), ('dice2', '4D')])
AttributeClause(source='Acumen 2D+2: survival 3D ', tokens=[('word', 'Acumen'), ('dice1', '2D+2'), ('punctuation', ':'), ('word', 'surv

Using this to process characters will yield numerous problems that need to be fixed manually.

## Step 3: Unpack the various kinds of clauses

The six base attributes will all have a common format.

```ebnf
    base : <attribute> <dice> [<punctuation>]? [<skill>]*

    skill : <word>+ dice [<punctuation>]?
```

The "other" attributes have unique formats, but follow a common pattern:

```ebnf
    other : <word>+ [<dice> | <number>] [<word> | <punctuation>]*
```

The optional attributes have two distinct forms: abilities and equipment.

```ebnf
    optional : <name> ":"? [<feature>]+

    feature : <word>+ <rank> [<word> | <number> | <punctuation>]*

    name : "Disadvantages" | "Advantages" | "Special Abilities" | "Natural Abilities"
```

```ebnf
    equipment : "Equipment" ":"? [<item>]+

    item : <word>+ "(" <word>+  <dice> ")" [<punctuation>]?
```


The target data structure is a collection of dataclasses that can be serialized into TOML (or JSON or what-have-you).

This structure is output from Phase 1 (Source Extraction) and input to Phase 2 (Character Generation).

In [7]:
from character_source import *

The `clause_factory()` function parses the tokenized text of each clause and creates a target data class object from it.

This will fail in places where the leading keyword is broken up by the OCR processing (e.g. `Disadvantage s`).
These, too, need to be tweaked manually.

In [8]:
other_lead_in = {"Strength", "Move", "Fate", "Character", "Body", "Wound"}

option_lead_in = {"Disadvantages", "Advantages", "Special", "Natural"}


def clause_factory(clause):

    def base_attribute_factory(word, clause):
        """
            Parse <attribute> : <word> <dice> <punctuation> <skill>+
            And <skill> : <word>+ <dice>
        """
        attribute = word
        d_label, dice = clause.tokens[1]
        if d_label not in {'dice1', 'dice2'}: raise SyntaxError(clause)
        skills_text = [[]]
        tail = deque(clause.tokens[2:])
        while tail:
            token = tail.popleft()
            match token:
                case ('punctuation', _):
                    continue
                case ('number1', _):
                    continue
                case ('word', skill):
                    skills_text[-1].append(skill)
                case ('dice1', skill_dice) | ('dice2', skill_dice):
                    skills_text[-1].append(skill_dice)
                    skills_text.append([])
                case _:
                    raise SyntaxError(clause)
        skills = [
            Skill(' '.join(label), dice)
            for *label, dice in skills_text[:-1]
        ]
        return Attribute(attribute, dice, skills)

    def other_attribute_factory(word, clause):
        """
            Parse <other> : <word> [<dice> | <number>] [<word> | <punctuation>]+
        """
        label = []
        value = ""
        suffix = []
        tail = deque(clause.tokens[:])
        while tail:
            token = tail.popleft()
            match token:
                case ('dice1', val) | ('dice2', val) | ('number1', val) | ('number2', val):
                    value = val
                case ('word', word):
                    if value:
                        suffix.append(word)
                    else:
                        label.append(word)
                case _:
                    _, token_text = token
                    suffix.append(token_text)
        return OtherAttribute(' '.join(label), value, ' '.join(suffix))

    def optional_attribute_factory(word, clause):
        """
            Parse <word>+ ":"? <ability>+
            and <ability> : <word>+ <rank> <anything>+ [";" | ","]
        """
        option_type = word
        start = 1
        while clause.tokens[start] in {('word', 'abilities'), ('word', 'Abilities'), ('punctuation', ':')}:
            start += 1
        abilities = []
        name, rank, details = [], "", []
        tail = deque(clause.tokens[start:])
        # Parse <word>+ <rank> <anything>+ [";" | ","]
        while tail:
            token = tail.popleft()
            match token:
                case ('punctuation', ';'):
                    abilities.append(Ability(' '.join(name), rank, ' '.join(details)))
                    name, rank, details = [], "", []
                case ('rank', rank_text):
                    if rank:
                        abilities.append(Ability(' '.join(name), rank, ' '.join(details)))
                        name, rank, details = [], "", []
                    rank = rank_text
                case ('dice1', dice) | ('dice2', dice):
                    details.append(dice)
                case _:
                    _, token_text = token
                    if rank:
                        details.append(token_text)
                    else:
                        name.append(token_text)
        if rank:
            abilities.append(Ability(' '.join(name), rank, ' '.join(details)))

        return Option(option_type, abilities)

    def equipment_factory(word, clause):
        """
            Parse "Equipment" ":"? <item>+
            And <item> : <word>+ "(" [<word> | <dice>] + ")" <punctuation>+
        """
        start = 1
        while clause.tokens[start] in {('punctuation', ':')}:
            start += 1
        items = []
        name, details = [], []
        tail = deque(clause.tokens[start:])
        # parse <word>+ "(" [<word> | <dice>] + ")" <punctuation>+
        while tail:
            token = tail.popleft()
            match token:
                case ('punctuation', ';'):
                    items.append(Item(' '.join(name), ' '.join(details)))
                    name, details = [], []
                case ('punctuation', '('):
                    details.append('(')
                case ('punctuation', ')'):
                    details.append(')')
                    items.append(Item(' '.join(name), ' '.join(details)))
                    name, details = [], []
                case _:
                    _, token_text = token
                    if details:
                        details.append(token_text)
                    else:
                        name.append(token_text)
        if name or details:
            items.append(Item(' '.join(name), ' '.join(details)))
        return Equipment(items)


    match clause.tokens[0]:
        case ('word', word) if word in base_attributes:
            return base_attribute_factory(word, clause)

        case ('word', word) if word in other_lead_in:
            return other_attribute_factory(word, clause)

        case ('word', word) if word in option_lead_in:
            return optional_attribute_factory(word, clause)

        case ('word', 'Equipment'):
            return equipment_factory(word, clause)

        case _:
            raise SyntaxError(f"initial word unknown: {clause}")

## Finally

We can explore the document tree looking for all character `.src` files.

In [9]:
def character_parse_iter(path):
    for character_text in character_text_iter(path):
        try:
            details = {
                clause.label: clause
                for clause in map(clause_factory, character_text.clause_iter())
            }
        except Exception:
            print(f"{p.parent.parent.name}/*/{p.name:22s}")
            print(f"  {character_text.name}")
            for clause in character_text.clause_iter():
                print(f"    {clause}")
                print(f"      {clause_factory(clause)}")
            raise
        yield Character(character_text.name, details)


character_paths = base_dir.glob("*/characters/*.src")
for p in character_paths:
    target = p.with_suffix(".toml")
    print(f"{p.parent.parent.name}/*/{p.name} -> {target.name}")

    target.write_text(
        tomli_w.dumps(
            {
                'characters': [
                    asdict(c) for c in character_parse_iter(p)
                ]
            }
        )
    )

Settlements/*/viking_warrior.src -> viking_warrior.toml
Settlements/*/philsopher.src -> philsopher.toml
Settlements/*/ladira_almer.src -> ladira_almer.toml
Settlements/*/mayan_priest.src -> mayan_priest.toml
Settlements/*/healfdene.src -> healfdene.toml
Settlements/*/cachamwri_cabry.src -> cachamwri_cabry.toml
Settlements/*/ori_swifthand.src -> ori_swifthand.toml
Settlements/*/ascara.src -> ascara.toml
Settlements/*/cire.src -> cire.toml
Settlements/*/dock_worker.src -> dock_worker.toml
Settlements/*/tiu_mali.src -> tiu_mali.toml
Settlements/*/nevest.src -> nevest.toml
Settlements/*/mukhtar.src -> mukhtar.toml
Settlements/*/cernay_avers.src -> cernay_avers.toml
Settlements/*/garvin_helot.src -> garvin_helot.toml
Settlements/*/namo.src -> namo.toml
Settlements/*/library_guard.src -> library_guard.toml
Settlements/*/balam_mis.src -> balam_mis.toml
Settlements/*/acolyte.src -> acolyte.toml
Settlements/*/kasen.src -> kasen.toml
Settlements/*/morcades.src -> morcades.toml
Settlements/*/beth

When the conversion to TOML runs error-free, it's time to switch to a different notebook to emit proper Python modules from the definitions.

This proceeds in three steps:

1. Load the TOML version of the text.
2. Create a `opend6_tools.Character` or `opend6_tools.Creature` object populated with appropriate details.
3. Use `opend6_tools.notebook_extract.ModuleWriter` to emit a proper character module.

In [10]:
Path.cwd()

PosixPath('/Users/slott/Documents/Hobbies/Gaming/OpenD6/world/fantasy_locations/notebooks')